### Rag Pipeline - Data Ingestion to Vectod DB Pipeline

In [36]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path



In [37]:
### Read all the PDF inside the directory
def process_all_pdfs(pdf_directory):
    """
    Process all PDF files in directory.
    """
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find all pdf recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f" ✓Loaded {len(documents)} pages")

        except Exception as e:
            print(f" ❌ Error : {e}")

    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

#Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 5 PDF files to process
Processing ../data/pdf/Python Interview.pdf
 ✓Loaded 72 pages
Processing ../data/pdf/Oops interview Question.pdf
 ✓Loaded 33 pages
Processing ../data/pdf/SQL Interview.pdf
 ✓Loaded 55 pages
Processing ../data/pdf/HR question.pdf
 ✓Loaded 34 pages
Processing ../data/pdf/DBMS Interview Question.pdf
 ✓Loaded 26 pages
Total documents loaded: 220


In [38]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m85', 'creator': 'Chromium', 'creationdate': '2022-06-16T08:18:52+00:00', 'moddate': '2022-06-16T08:18:52+00:00', 'source': '../data/pdf/Python Interview.pdf', 'total_pages': 72, 'page': 0, 'page_label': '1', 'source_file': 'Python Interview.pdf', 'file_type': 'pdf'}, page_content='Python Interview Questions\nTo view the live version of the\npage, click here.\n© Copyright by Interviewbit'),
 Document(metadata={'producer': 'Skia/PDF m85', 'creator': 'Chromium', 'creationdate': '2022-06-16T08:18:52+00:00', 'moddate': '2022-06-16T08:18:52+00:00', 'source': '../data/pdf/Python Interview.pdf', 'total_pages': 72, 'page': 1, 'page_label': '2', 'source_file': 'Python Interview.pdf', 'file_type': 'pdf'}, page_content="If you're a Python Developer looking to learn more, the guidance of an\nexpert might complement the insights from the book. If you're\ninterested, feel free to register for our FREE upcoming Masterclass,\nwhere you can interact with a top 

### Chunking

In [39]:
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [40]:
chunks = split_documents(all_pdf_documents)
chunks

Split 220 documents into 371 chunks

Example chunk:
Content: Python Interview Questions
To view the live version of the
page, click here.
© Copyright by Interviewbit...
Metadata: {'producer': 'Skia/PDF m85', 'creator': 'Chromium', 'creationdate': '2022-06-16T08:18:52+00:00', 'moddate': '2022-06-16T08:18:52+00:00', 'source': '../data/pdf/Python Interview.pdf', 'total_pages': 72, 'page': 0, 'page_label': '1', 'source_file': 'Python Interview.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m85', 'creator': 'Chromium', 'creationdate': '2022-06-16T08:18:52+00:00', 'moddate': '2022-06-16T08:18:52+00:00', 'source': '../data/pdf/Python Interview.pdf', 'total_pages': 72, 'page': 0, 'page_label': '1', 'source_file': 'Python Interview.pdf', 'file_type': 'pdf'}, page_content='Python Interview Questions\nTo view the live version of the\npage, click here.\n© Copyright by Interviewbit'),
 Document(metadata={'producer': 'Skia/PDF m85', 'creator': 'Chromium', 'creationdate': '2022-06-16T08:18:52+00:00', 'moddate': '2022-06-16T08:18:52+00:00', 'source': '../data/pdf/Python Interview.pdf', 'total_pages': 72, 'page': 1, 'page_label': '2', 'source_file': 'Python Interview.pdf', 'file_type': 'pdf'}, page_content="If you're a Python Developer looking to learn more, the guidance of an\nexpert might complement the insights from the book. If you're\ninterested, feel free to register for our FREE upcoming Masterclass,\nwhere you can interact with a top 

### Embedding and VectorStore DB

In [41]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [42]:
class EmbeddingManager:
    """Handles documents ebedding generation using sentence-transformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initializes the EmbeddingManager with a specified model name.
        
        Args:
            model_name: HuggingFace model name for sentence-transformers."
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads the sentence-transformers model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully.Embedding dimensions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generates embeddings for a list of texts.
        
        Args:
            texts: List of text strings to embed.

        Returns:
            A numpy array of embeddings with shape (len(texts), embedding_dimension).
        """
        if not self.model:
            raise ValueError("Model is not loaded. Call _load_model() first.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Embeddings generated successfully. Shape: {embeddings.shape}")
        return embeddings

In [43]:
##initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully.Embedding dimensions: 384


### Vector Store

In [44]:
class vectorStore:
    """Handles storage and retrieval of document embeddings using ChromaDB."""

    def __init__(self, collection_name: str = "pdf_embeddings", persist_directory: str = "../data/vector_store"):
        """Initializes the VectorStore with a specified collection name and persistence directory.
        
        Args:
            collection_name: Name of the ChromaDB collection to store embeddings.
            persist_directory: Directory to persist the vector store.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initializes the chromaDB client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create the collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "PDF document embeddings for RAG"})
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing document in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_document(self, documents, embeddings):
        """
        Adds documents and their corresponding embeddings to the VectorStore.
        
        Args:
            documents: List of Langchain Document.
            embeddings: List of embeddings corresponding to the documents.
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")

        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_texts = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            #Generate Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #Document Content
            documents_texts.append(doc.page_content)

            #Embedding
            embedding_list.append(embedding.tolist())

        #Add to Collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embedding_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
    
vectorstore = vectorStore()
vectorstore

Vector store initialized. Collection: pdf_embeddings
Existing document in collection: 371


In [45]:
chunks

[Document(metadata={'producer': 'Skia/PDF m85', 'creator': 'Chromium', 'creationdate': '2022-06-16T08:18:52+00:00', 'moddate': '2022-06-16T08:18:52+00:00', 'source': '../data/pdf/Python Interview.pdf', 'total_pages': 72, 'page': 0, 'page_label': '1', 'source_file': 'Python Interview.pdf', 'file_type': 'pdf'}, page_content='Python Interview Questions\nTo view the live version of the\npage, click here.\n© Copyright by Interviewbit'),
 Document(metadata={'producer': 'Skia/PDF m85', 'creator': 'Chromium', 'creationdate': '2022-06-16T08:18:52+00:00', 'moddate': '2022-06-16T08:18:52+00:00', 'source': '../data/pdf/Python Interview.pdf', 'total_pages': 72, 'page': 1, 'page_label': '2', 'source_file': 'Python Interview.pdf', 'file_type': 'pdf'}, page_content="If you're a Python Developer looking to learn more, the guidance of an\nexpert might complement the insights from the book. If you're\ninterested, feel free to register for our FREE upcoming Masterclass,\nwhere you can interact with a top 

In [46]:
### Convert text to Embeddings
texts = [doc.page_content for doc in chunks]

###Generatie the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

###Store in the vectorDB
vectorstore.add_document(chunks, embeddings)


Generating embeddings for 371 texts...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches: 100%|██████████| 12/12 [00:01<00:00,  7.87it/s]


Embeddings generated successfully. Shape: (371, 384)
Adding 371 documents to the vector store...
Successfully added 371 documents to the vector store.
Total documents in collection after addition: 742


### Retriver pipeline from Vector Store

In [52]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: vectorstore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [53]:
rag_retriever

In [54]:
rag_retriever.retrieve("what is Python Interview is all you need")

Retrieving documents for query: 'what is Python Interview is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.34s/it]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_2aba9a46_119',
  'content': 'Python Interview Questions\nIn this article, we have seen commonly asked interview questions for a python\ndeveloper. These questions along with regular problem practice sessions will help\nyou crack any python based interviews. Over the years, python has gained a lot of\npopularity amongst the developer’s community due to its simplicity and ability to\nsupport powerful computations. Due to this, the demand for good python developers\nis ever-growing. Nevertheless, to mention, the perks of being a python developer are\nreally good. Along with theoretical knowledge in python, there is an emphasis on the\nability to write good quality code as well. So, keep learning and keep practicing\nproblems and without a doubt, you can crack any interviews.\nImportant Resources:\nPython Basic Programs\nPython MCQ\nPython Commands\nPython Developer Resume\nPython Projects\nDiﬀerence Between Python 2 and 3\nPython Frameworks\nPython Documentation\nNumpy Tutori

In [57]:
rag_retriever.retrieve("explain outer join")

Retrieving documents for query: 'explain outer join'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.32it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_4e1222c8_194',
  'content': 'SQL Interview Questions\nSELECT *\nFROM Table_A \nJOIN Table_B; \nSELECT *\nFROM Table_A \nINNER JOIN Table_B;\nLEFT (OUTER) JOIN: Retrieves all the records/rows from the le\x00 and the\nmatched records/rows from the right table.\nSELECT *\nFROM Table_A A \nLEFT JOIN Table_B B \nON A.col = B.col;\nRIGHT (OUTER) JOIN: Retrieves all the records/rows from the right and the\nmatched records/rows from the le\x00 table.\nSELECT *\nFROM Table_A A \nRIGHT JOIN Table_B B \nON A.col = B.col;\nFULL (OUTER) JOIN: Retrieves all the records where there is a match in either\nthe le\x00 or right table.\nSELECT *\nFROM Table_A A \nFULL JOIN Table_B B \nON A.col = B.col;\n12.\xa0\xa0\xa0What is a Self-Join?\nA self JOIN is a case of regular join where a table is joined to itself based on some\nrelation between its own column(s). Self-join uses the INNER JOIN or LEFT JOIN\nclause and a table alias is used to assign diﬀerent names to the table within the\nquery.\n

### RAG Pipeline- VectorDB To LLM Output Generation

In [66]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [70]:
answer=rag_simple("types of join",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'types of join'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.20it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)


The four types of SQL joins are:

1. **INNER JOIN** – returns only rows with matching values in both tables.  
2. **LEFT (OUTER) JOIN** – returns all rows from the left table and the matching rows from the right table (nulls for non‑matches).  
3. **RIGHT (OUTER) JOIN** – returns all rows from the right table and the matching rows from the left table.  
4. **FULL (OUTER) JOIN** – returns all rows that have a match in either the left or the right table (including non‑matching rows from both sides).


In [68]:
answer=rag_simple("type of laptop",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'type of laptop'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.17it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 0 documents (after filtering)
No relevant context found to answer the question.


### Enhanced RAG Pipeline Features

In [71]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("what is database?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what is database?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.75s/it]


Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: A database is an organized, consistent, and logical collection of data that can be easily updated, accessed, and managed—typically stored as tables (or other database objects) containing rows (records/tuples) and columns (attributes).
Sources: [{'source': 'DBMS Interview Question.pdf', 'page': 4, 'score': 0.44562017917633057, 'preview': 'DBMS Interview Questions\nRDBMS stands for Relational Database Management System and was introduced in\nthe 1970s to access and store data more eﬀiciently than DBMS. RDBMS stores data in\nthe form of tables as compared to DBMS which stores data as files. Storing data as\nrows and columns makes it easier...'}, {'source': 'DBMS Interview Question.pdf', 'page': 4, 'score': 0.44562017917633057, 'preview': 'DBMS Interview Questions\nRDBMS stands for Relational Database Management System and was introduced in\nthe 1970s to access and store data more eﬀiciently 

In [73]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("data types in python?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'data types in python?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.91it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Python Interview Questions
There are several built-in data types in Python. Although, Python doesn't require data
types to be defined explicitly during variable declarations type errors are likely to
occur if the knowledge of data types and their comp

atibility with each other are
neglected. Python provides type()  and isinstance()  functions to check the type
of these variables. These data types can be grouped into the following categories-
None Type:
None  keyword represents the null values in Python. Boolean equality
operation can be performed using these NoneType objects.
Class Name Description
NoneType Represents the NULL values in Python.
Numeric Types:
There are three distinct numeric types - integers, floating-point numbers, and
complex numbers. Additionally, booleans are a sub-type of integers.
Class Name Description
int Stores integer literals including hex, octal and
binary numbers as integers
float
Stores literals containing decimal values

Python Interview Questions
There are several built-in data types in Python. Although, Python doesn't require data
types to be defined explicitly during variable declarations type errors are likely to
occur if the knowledge of data types and their compatibility with each other are
negl